# Heart Disease Prediction: Robust Classification & Probability Calibration
**Author**: Portfolio Project (Senior Data Science Track)  
**Dataset**: UCI Machine Learning Repository — Cleveland Clinic Database (Janosi et al., 1988)  
**License**: Creative Commons Attribution 4.0 International (CC BY 4.0)

> **Important Clinical & Ethical Disclaimer**:
> This machine learning model is developed strictly for **educational and scientific research purposes**. It is **not** a validated diagnostic tool, medical device, or clinical decision support software. High predictive accuracy on small retrospective datasets must not be conflated with prospective clinical utility. Real clinical diagnoses require comprehensive medical evaluation, lab work, physician judgment, and institutional ethical approvals.


In [1]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", message="Found unknown categories.*")
warnings.filterwarnings("ignore", category=FutureWarning)

# Add project root to sys.path
sys.path.append(os.path.abspath(".."))
from src.data_loader import load_heart_data, get_train_test_split, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from src.pipeline import build_full_pipeline
from src.evaluate import compute_classification_metrics

# Set visualization styles
sns.set_theme(style="whitegrid")
plt.rcParams["font.size"] = 10


## 1. Data Ingestion & Integrity Inspection
The dataset contains 303 patient records with 14 clinical features. We verify missingness patterns, datatypes, and class balance.
Target is binarized: 0 (angiographic disease absent) vs 1 (angiographic disease present, >50% diameter narrowing).


In [2]:
df = load_heart_data('../data/processed_cleveland.csv')
print(f"Dataset shape: {df.shape}")
print(df.head())
print("\nClass Distribution:")
print(df['target'].value_counts(normalize=True).rename({0: 'Absence (0)', 1: 'Presence (1)'}))


Dataset shape: (303, 14)
    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   
4  41.0  0.0  2.0     130.0  204.0  0.0      2.0    172.0    0.0      1.4   

   slope   ca  thal  target  
0    3.0  0.0   6.0       0  
1    2.0  3.0   3.0       1  
2    2.0  2.0   7.0       1  
3    3.0  0.0   3.0       0  
4    1.0  0.0   3.0       0  

Class Distribution:
target
Absence (0)     0.541254
Presence (1)    0.458746
Name: proportion, dtype: float64


## 2. Exploratory Data Analysis (EDA)
We inspect key clinical variables such as `thalach` (max heart rate), `oldpeak` (exercise ST depression), `age`, and `chol` against the target.


In [3]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
sns.boxplot(data=df, x='target', y='thalach', ax=axes[0], palette=['#3470a3', '#e05a47'])
axes[0].set_title('Max Heart Rate (thalach) by Target')

sns.boxplot(data=df, x='target', y='oldpeak', ax=axes[1], palette=['#3470a3', '#e05a47'])
axes[1].set_title('ST Depression (oldpeak) by Target')

sns.scatterplot(data=df, x='age', y='thalach', hue='target', ax=axes[2], palette=['#3470a3', '#e05a47'], alpha=0.8)
axes[2].set_title('Age vs Max Heart Rate')
plt.tight_layout()
plt.savefig('../reports/figures/notebook_eda_sample.png', dpi=150)
plt.close()
print("Saved notebook EDA sample figure.")


Saved notebook EDA sample figure.


## 3. Leakage-Free Preprocessing Pipeline & Stratified Validation
To avoid data leakage, all imputation and scaling estimators are fit strictly inside cross-validation training folds.
We compare:
1. **Dummy Baseline** (stratified random)
2. **Logistic Regression** (L2 regularized)
3. **Random Forest Classifier**
4. **Gradient Boosting Classifier**


In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

X_train, X_test, y_train, y_test = get_train_test_split(df, test_size=0.2, random_state=42)

models = {
    'Dummy Baseline': DummyClassifier(strategy='stratified', random_state=42),
    'Logistic Regression': LogisticRegression(C=0.5, solver='lbfgs', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=4, min_samples_split=4, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, max_depth=3, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_records = []

for name, clf in models.items():
    pipe = build_full_pipeline(clf)
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=['roc_auc', 'f1', 'recall', 'precision'], n_jobs=-1)
    cv_records.append({
        'Model': name,
        'CV ROC-AUC': f"{np.mean(scores['test_roc_auc']):.3f} +/- {np.std(scores['test_roc_auc']):.3f}",
        'CV F1': f"{np.mean(scores['test_f1']):.3f} +/- {np.std(scores['test_f1']):.3f}",
        'CV Recall': f"{np.mean(scores['test_recall']):.3f} +/- {np.std(scores['test_recall']):.3f}",
        'CV Precision': f"{np.mean(scores['test_precision']):.3f} +/- {np.std(scores['test_precision']):.3f}"
    })

print(pd.DataFrame(cv_records).to_markdown(index=False))


| Model               | CV ROC-AUC      | CV F1           | CV Recall       | CV Precision    |
|:--------------------|:----------------|:----------------|:----------------|:----------------|
| Dummy Baseline      | 0.515 +/- 0.077 | 0.433 +/- 0.093 | 0.396 +/- 0.084 | 0.478 +/- 0.105 |
| Logistic Regression | 0.900 +/- 0.016 | 0.815 +/- 0.014 | 0.792 +/- 0.063 | 0.850 +/- 0.076 |
| Random Forest       | 0.881 +/- 0.027 | 0.771 +/- 0.014 | 0.729 +/- 0.043 | 0.825 +/- 0.060 |
| Gradient Boosting   | 0.858 +/- 0.023 | 0.747 +/- 0.048 | 0.720 +/- 0.055 | 0.776 +/- 0.042 |


## 4. Unbiased Holdout Test Set Evaluation
We evaluate on the 61 held-out test patients that were never exposed to training or hyperparameter selection.


In [5]:
test_records = []
fitted_models = {}

for name, clf in models.items():
    pipe = build_full_pipeline(clf)
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe
    
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, 'predict_proba') else None
    
    metrics = compute_classification_metrics(y_test.values, y_pred, y_proba)
    test_records.append({
        'Model': name,
        'Accuracy': metrics['accuracy'],
        'Precision': metrics['precision'],
        'Recall': metrics['recall_sensitivity'],
        'Specificity': metrics['specificity'],
        'F1 Score': metrics['f1_score'],
        'ROC-AUC': metrics.get('roc_auc', np.nan),
        'Brier Score': metrics.get('brier_score', np.nan)
    })

print(pd.DataFrame(test_records).round(3).to_markdown(index=False))


| Model               |   Accuracy |   Precision |   Recall |   Specificity |   F1 Score |   ROC-AUC |   Brier Score |
|:--------------------|-----------:|------------:|---------:|--------------:|-----------:|----------:|--------------:|
| Dummy Baseline      |      0.361 |       0.28  |    0.25  |         0.455 |      0.264 |     0.352 |         0.639 |
| Logistic Regression |      0.885 |       0.862 |    0.893 |         0.879 |      0.877 |     0.961 |         0.084 |
| Random Forest       |      0.885 |       0.862 |    0.893 |         0.879 |      0.877 |     0.944 |         0.112 |
| Gradient Boosting   |      0.869 |       0.857 |    0.857 |         0.879 |      0.857 |     0.934 |         0.104 |


## 5. Model Interpretability & Calibration
Logistic regression provides high discrimination (ROC-AUC 0.961 on test holdout) and calibrated posterior probabilities (Brier score 0.084).


In [6]:
from sklearn.inspection import permutation_importance

champion = fitted_models['Logistic Regression']
perm = permutation_importance(champion, X_test, y_test, n_repeats=20, random_state=42, scoring='roc_auc')

perm_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance Mean': perm.importances_mean,
    'Importance Std': perm.importances_std
}).sort_values('Importance Mean', ascending=True)

plt.figure(figsize=(7, 4.5))
plt.barh(perm_df['Feature'], perm_df['Importance Mean'], xerr=perm_df['Importance Std'], color='#2b5c8f', capsize=3)
plt.xlabel('Mean Decrease in ROC-AUC')
plt.title('Permutation Feature Importance on Held-Out Test Set')
plt.tight_layout()
plt.savefig('../reports/figures/notebook_importance_sample.png', dpi=150)
plt.close()
print("Saved feature importance figure.")


Saved feature importance figure.


## 6. Scientific Discussion: Limitations, Uncertainty & Leakage Considerations

### 1. Data Leakage Prevention
- **Imputation & Standardization**: Executed strictly within training folds via `Pipeline` and `ColumnTransformer`. No test-set statistics inform missing value replacement or feature scaling.
- **Stratified Partitioning**: The 80/20 train/test split maintains identical target proportion (45.9% positive).

### 2. Dataset Limitations
- **Sample Size**: N = 303 is modest for tabular modeling, causing wider confidence intervals on subset metrics.
- **Demographic Bias**: The original Cleveland Clinic cohort is predominantly male (68% male, 32% female), which risks demographic bias in coronary symptom presentation.
- **Historical Context**: The data was recorded in the late 1980s. Modern biomarkers (e.g., high-sensitivity troponin, coronary CT angiography, advanced echocardiography) are not present.

### 3. Clinical Limitations & Uncertainty
- **False Negative Severity**: In cardiac screening, missing an active CAD case (false negative) carries far greater clinical risk than a false positive. Logistic regression achieves 89.3% sensitivity (recall) at threshold 0.5. Lowering the operational decision threshold (e.g., to 0.35) can further elevate sensitivity in high-risk triage settings.
- **Spectrum Bias**: Retrospective clinic patients referred for coronary angiography inherently carry higher pre-test probability than an unselected general outpatient population.
